In [ ]:
import pandas as pd
import numpy as np
import os

# --- CONFIGURATION ---
# Define the order of metrics and which CSV they come from
# keys: 'Display Name', 'CSV Metric Name', 'Source Index' (0=Perf CSV, 1=Interp CSV)
METRIC_CONFIG = [
    {'disp': 'CAV Acc', 'csv_name': 'Accuracy',          'source': 0},
    {'disp': 'Cos Sim', 'csv_name': 'Cosine Similarity', 'source': 1},
    {'disp': 'Sens',    'csv_name': 'Sensitivity',       'source': 1},
    {'disp': 'TCAV',    'csv_name': 'TCAV Score',        'source': 1}
]

def format_p_value(val):
    """Formats p-value for LaTeX."""
    try:
        p = float(val)
        if p < 0.001:
            return r"<.001"
        return f"{p:.3f}"
    except (ValueError, TypeError):
        return "-"

def format_float(val):
    """Formats standard floats."""
    try:
        return f"{float(val):.2f}"
    except (ValueError, TypeError):
        return "-"

def get_layer_data(df, metric_name, col_name, fmt_func):
    """
    Extracts data for layers 0-11 for a specific metric.
    Returns a list of 12 formatted strings.
    """
    results = []
    # Filter dataframe for the specific metric
    # We assume the CSV has a 'Metric' column and 'Layer' column (0 to 11)
    subset = df[df['Metric'] == metric_name]
    
    for layer in range(12):
        row = subset[subset['Layer'] == layer]
        if not row.empty:
            val = row.iloc[0][col_name]
            results.append(fmt_func(val))
        else:
            results.append("-")
    return results

def generate_table(experiments):
    """
    experiments: List of tuples ("Concept Name", "path_to_acc_csv", "path_to_tcav_csv")
    """
    latex_lines = []
    
    # --- 1. Header ---
    header = r"""\begin{table}[ht]
\centering
\resizebox{\textwidth}{!}{%
\begin{tabular}{llcccccccccccc}
\toprule
\textbf{Concept} & \textbf{Metric} & \textbf{L0} & \textbf{L1} & \textbf{L2} & \textbf{L3} & \textbf{L4} & \textbf{L5} & \textbf{L6} & \textbf{L7} & \textbf{L8} & \textbf{L9} & \textbf{L10} & \textbf{L11} \\
\midrule"""
    latex_lines.append(header)

    # --- 2. Process Null (Using the first experiment as the source of truth) ---
    if not experiments:
        print("No experiments provided.")
        return

    first_concept = experiments[0]
    dfs = [pd.read_csv(first_concept[1]), pd.read_csv(first_concept[2])]
    
    latex_lines.append(r"\multirow{4}{*}{Null}")
    
    for m in METRIC_CONFIG:
        # Load the correct DF (Performance or Interpretability)
        current_df = dfs[m['source']]
        
        # Get Null Means
        null_vals = get_layer_data(current_df, m['csv_name'], 'Null_Mean', format_float)
        latex_lines.append(f" & {m['disp']} & " + " & ".join(null_vals) + r" \\")
        if m != METRIC_CONFIG[-1]:
            # Divider
            latex_lines.append(r" \cmidrule(l){2-14}")
    
    latex_lines.append(r"\midrule")

    # --- 3. Process Each Concept ---
    for concept_name, acc_path, tcav_path in experiments:
        # Load Dataframes
        try:
            df_perf = pd.read_csv(acc_path)
            df_interp = pd.read_csv(tcav_path)
            dfs = [df_perf, df_interp]
        except FileNotFoundError as e:
            print(f"Error loading files for {concept_name}: {e}")
            continue

        # Clean concept name for LaTeX
        clean_name = concept_name.replace("_", r"\_")
        
        # Concept Multirow Header
        latex_lines.append(r"\multirow{12}{*}{\shortstack[l]{" + clean_name + r"}}")
        
        for i, m in enumerate(METRIC_CONFIG):
            current_df = dfs[m['source']]
            
            # 1. Real Mean
            real_vals = get_layer_data(current_df, m['csv_name'], 'Real_Mean', format_float)
            latex_lines.append(f" & {m['disp']} & " + " & ".join(real_vals) + r" \\")
            
            # 2. P-Value (Grey, Footnotesize)
            p_vals = get_layer_data(current_df, m['csv_name'], 'P_Adjusted', format_p_value)
            p_str = " & ".join([fr"\footnotesize\color{{gray}}{v}" for v in p_vals])
            latex_lines.append(r" & \footnotesize\color{gray}\textit{p-val} & " + p_str + r" \\")
            
            # 3. Cohen's d (Grey, Footnotesize)
            d_vals = get_layer_data(current_df, m['csv_name'], 'Effect_Size_Cohen_d', format_float)
            d_str = " & ".join([fr"\footnotesize\color{{gray}}{v}" for v in d_vals])
            latex_lines.append(r" & \footnotesize\color{gray}\textit{Cohen's d} & " + d_str + r" \\")
            
            # Divider (unless it's the last metric)
            if i < len(METRIC_CONFIG) - 1:
                latex_lines.append(r" \cmidrule(l){2-14}")
        
        # Add a separator between concepts (optional, but good for readability)
        if experiments.index((concept_name, acc_path, tcav_path)) < len(experiments) - 1:
            latex_lines.append(r"\midrule")

    # --- 4. Footer ---
    footer = r"""\bottomrule
\end{tabular}%
}
\caption{Summary of metrics. Statistics (p-val, Cohen's d) are shown in grey.}
\label{tab:metrics_summary}
\end{table}"""
    latex_lines.append(footer)
    
    # --- 5. Output ---
    full_latex = "\n".join(latex_lines)
    
    # Save to file
    with open("final_metrics_table.tex", "w") as f:
        f.write(full_latex)
    
    print("LaTeX table generated successfully: final_metrics_table.tex")
    print("-" * 20)
    print(full_latex)

# ==========================================
#              USER INPUTS
# ==========================================

# List of tuples: (Concept Name, Path to Acc/AUC CSV, Path to TCAV/Sens CSV)
EXPERIMENTS = [
    (
        "Circling Open", 
        "/Volumes/T7/Concepts/TCAV/circling/concepts/circling_open/3_real_vs_null_cavs/stats_summary_concept_set_filter.csv",  # From Script 2 (Acc, AUC)
        "/Volumes/T7/Concepts/TCAV/circling/concepts/circling_open/5_real_vs_null_tcav_scores/stats_summary_concept_set_filter.csv",  # From Script 1 (TCAV, Sens)
    ),
    (
        "Circling Closed", 
        "/Volumes/T7/Concepts/TCAV/circling/concepts/circling_closed/3_real_vs_null_cavs/stats_summary_concept_set_filter.csv",  # From Script 2 (Acc, AUC)
        "/Volumes/T7/Concepts/TCAV/circling/concepts/circling_closed/5_real_vs_null_tcav_scores/stats_summary_concept_set_filter.csv",  # From Script 1 (TCAV, Sens)
    ),
    (
        "MG Instructed Coordination",
        "/Volumes/T7/Concepts/TCAV/circling/concepts/mg_coordination/3_real_vs_null_cavs/stats_summary_concept_set_filter.csv",  # From Script 2 (Acc, AUC)
        "/Volumes/T7/Concepts/TCAV/circling/concepts/mg_coordination/5_real_vs_null_tcav_scores/stats_summary_concept_set_filter.csv",  # From Script 1 (TCAV, Sens)
    ),
    (
        "MG Spontaneous Coordination",
        "/Volumes/T7/Concepts/TCAV/circling/concepts/mg_spontaneous/3_real_vs_null_cavs/stats_summary_concept_set_filter.csv",  # From Script 2 (Acc, AUC)
        "/Volumes/T7/Concepts/TCAV/circling/concepts/mg_spontaneous/5_real_vs_null_tcav_scores/stats_summary_concept_set_filter.csv",  # From Script 1 (TCAV, Sens)
    ),

    (
        "MG Solo",
        "/Volumes/T7/Concepts/TCAV/circling/concepts/mg_solo/3_real_vs_null_cavs/stats_summary_concept_set_filter.csv",  # From Script 2 (Acc, AUC)
        "/Volumes/T7/Concepts/TCAV/circling/concepts/mg_solo/5_real_vs_null_tcav_scores/stats_summary_concept_set_filter.csv",  # From Script 1 (TCAV, Sens)
    ),
]

if __name__ == "__main__":
    generate_table(EXPERIMENTS)

In [ ]:
import pandas as pd
import numpy as np
import os

# --- CONFIGURATION ---
# Define the order of metrics and which CSV they come from
# keys: 'Display Name', 'CSV Metric Name', 'Source Index' (0=Perf CSV, 1=Interp CSV)
METRIC_CONFIG = [
    {'disp': 'CAV Acc', 'csv_name': 'Accuracy',          'source': 0},
    {'disp': 'Cos Sim', 'csv_name': 'Cosine Similarity', 'source': 1},
    {'disp': 'Sens',    'csv_name': 'Sensitivity',       'source': 1},
    {'disp': 'TCAV',    'csv_name': 'TCAV Score',        'source': 1}
]

def format_p_value(val):
    """Formats p-value for LaTeX."""
    try:
        p = float(val)
        if p < 0.001:
            return r"<.001"
        return f"{p:.3f}"
    except (ValueError, TypeError):
        return "-"

def format_float(val):
    """Formats standard floats."""
    try:
        return f"{float(val):.3f}"
    except (ValueError, TypeError):
        return "-"

def get_layer_data(df, metric_name, col_name, fmt_func):
    """
    Extracts data for layers 0-11 for a specific metric.
    Returns a list of 12 formatted strings.
    """
    results = []
    # Filter dataframe for the specific metric
    # We assume the CSV has a 'Metric' column and 'Layer' column (0 to 11)
    subset = df[df['Metric'] == metric_name]
    
    for layer in range(12):
        row = subset[subset['Layer'] == layer]
        if not row.empty:
            val = row.iloc[0][col_name]
            results.append(fmt_func(val))
        else:
            results.append("-")
    return results

def generate_table(experiments):
    """
    experiments: List of tuples ("Concept Name", "path_to_acc_csv", "path_to_tcav_csv")
    """
    latex_lines = []
    
    # --- 1. Header ---
    header = r"""\begin{table}[ht]
\centering
\resizebox{\textwidth}{!}{%
\begin{tabular}{llcccccccccccc}
\toprule
\textbf{Concept} & \textbf{Metric} & \textbf{L0} & \textbf{L1} & \textbf{L2} & \textbf{L3} & \textbf{L4} & \textbf{L5} & \textbf{L6} & \textbf{L7} & \textbf{L8} & \textbf{L9} & \textbf{L10} & \textbf{L11} \\
\midrule"""
    latex_lines.append(header)

    # --- 2. Process Null (Using the first experiment as the source of truth) ---
    if not experiments:
        print("No experiments provided.")
        return

    first_concept = experiments[0]
    dfs = [pd.read_csv(first_concept[1]), pd.read_csv(first_concept[2])]
    
    latex_lines.append(r"\multirow{4}{*}{Null}")
    
    for m in METRIC_CONFIG:
        # Load the correct DF (Performance or Interpretability)
        current_df = dfs[m['source']]
        
        # Get Null Means
        null_vals = get_layer_data(current_df, m['csv_name'], 'Null_Mean', format_float)
        latex_lines.append(f" & {m['disp']} & " + " & ".join(null_vals) + r" \\")
        if m != METRIC_CONFIG[-1]:
            # Divider
            latex_lines.append(r" \cmidrule(l){2-14}")
    
    latex_lines.append(r"\midrule")

    # --- 3. Process Each Concept ---
    for concept_name, acc_path, tcav_path in experiments:
        # Load Dataframes
        try:
            df_perf = pd.read_csv(acc_path)
            df_interp = pd.read_csv(tcav_path)
            dfs = [df_perf, df_interp]
        except FileNotFoundError as e:
            print(f"Error loading files for {concept_name}: {e}")
            continue

        # Clean concept name for LaTeX
        clean_name = concept_name.replace("_", r"\_")
        
        # Concept Multirow Header
        latex_lines.append(r"\multirow{12}{*}{\shortstack[l]{" + clean_name + r"}}")
        
        for i, m in enumerate(METRIC_CONFIG):
            current_df = dfs[m['source']]
            
            # 1. Real Mean
            real_vals = get_layer_data(current_df, m['csv_name'], 'Real_Mean', format_float)
            latex_lines.append(f" & {m['disp']} & " + " & ".join(real_vals) + r" \\")
            
            # 2. P-Value (Grey, Footnotesize)
            p_vals = get_layer_data(current_df, m['csv_name'], 'P_Adjusted', format_p_value)
            p_str = " & ".join([fr"\footnotesize\color{{gray}}{v}" for v in p_vals])
            latex_lines.append(r" & \footnotesize\color{gray}\textit{p-val} & " + p_str + r" \\")
            
            # 3. Cohen's d (Grey, Footnotesize)
            d_vals = get_layer_data(current_df, m['csv_name'], 'Effect_Size_Cohen_d', format_float)
            d_str = " & ".join([fr"\footnotesize\color{{gray}}{v}" for v in d_vals])
            latex_lines.append(r" & \footnotesize\color{gray}\textit{Cohen's d} & " + d_str + r" \\")
            
            # Divider (unless it's the last metric)
            if i < len(METRIC_CONFIG) - 1:
                latex_lines.append(r" \cmidrule(l){2-14}")
        
        # Add a separator between concepts (optional, but good for readability)
        if experiments.index((concept_name, acc_path, tcav_path)) < len(experiments) - 1:
            latex_lines.append(r"\midrule")

    # --- 4. Footer ---
    footer = r"""\bottomrule
\end{tabular}%
}
\caption{Summary of metrics. Statistics (p-val, Cohen's d) are shown in grey.}
\label{tab:metrics_summary}
\end{table}"""
    latex_lines.append(footer)
    
    # --- 5. Output ---
    full_latex = "\n".join(latex_lines)
    
    # Save to file
    with open("final_metrics_table.tex", "w") as f:
        f.write(full_latex)
    
    print("LaTeX table generated successfully: final_metrics_table.tex")
    print("-" * 20)
    print(full_latex)

# ==========================================
#              USER INPUTS
# ==========================================

# List of tuples: (Concept Name, Path to Acc/AUC CSV, Path to TCAV/Sens CSV)
EXPERIMENTS = [
    (
        "Circling Open", 
        "/Volumes/T7/Concepts/TCAV/circling/concepts/circling_open/3_real_vs_null_cavs/stats_summary_concept_set_filter.csv",  # From Script 2 (Acc, AUC)
        "/Volumes/T7/Concepts/TCAV/circling/concepts/circling_open/5_real_vs_null_tcav_scores/stats_summary_concept_set_filter.csv",  # From Script 1 (TCAV, Sens)
    ),
    (
        "Circling Closed", 
        "/Volumes/T7/Concepts/TCAV/circling/concepts/circling_closed/3_real_vs_null_cavs/stats_summary_concept_set_filter.csv",  # From Script 2 (Acc, AUC)
        "/Volumes/T7/Concepts/TCAV/circling/concepts/circling_closed/5_real_vs_null_tcav_scores/stats_summary_concept_set_filter.csv",  # From Script 1 (TCAV, Sens)
    ),
    (
        "MG Instructed Coordination",
        "/Volumes/T7/Concepts/TCAV/circling/concepts/mg_coordination/3_real_vs_null_cavs/stats_summary_concept_set_filter.csv",  # From Script 2 (Acc, AUC)
        "/Volumes/T7/Concepts/TCAV/circling/concepts/mg_coordination/5_real_vs_null_tcav_scores/stats_summary_concept_set_filter.csv",  # From Script 1 (TCAV, Sens)
    ),
    (
        "MG Spontaneous Coordination",
        "/Volumes/T7/Concepts/TCAV/circling/concepts/mg_spontaneous/3_real_vs_null_cavs/stats_summary_concept_set_filter.csv",  # From Script 2 (Acc, AUC)
        "/Volumes/T7/Concepts/TCAV/circling/concepts/mg_spontaneous/5_real_vs_null_tcav_scores/stats_summary_concept_set_filter.csv",  # From Script 1 (TCAV, Sens)
    ),

    (
        "MG Solo",
        "/Volumes/T7/Concepts/TCAV/circling/concepts/mg_solo/3_real_vs_null_cavs/stats_summary_concept_set_filter.csv",  # From Script 2 (Acc, AUC)
        "/Volumes/T7/Concepts/TCAV/circling/concepts/mg_solo/5_real_vs_null_tcav_scores/stats_summary_concept_set_filter.csv",  # From Script 1 (TCAV, Sens)
    ),
]

if __name__ == "__main__":
    generate_table(EXPERIMENTS)

In [ ]:
import pandas as pd
import numpy as np
import os

# --- CONFIGURATION ---
# Define the order of metrics and which CSV they come from
# keys: 'Display Name', 'CSV Metric Name', 'Source Index' (0=Perf CSV, 1=Interp CSV)
METRIC_CONFIG = [
    {'disp': 'CAV Acc', 'csv_name': 'Accuracy',          'source': 0},
    {'disp': 'Cos Sim', 'csv_name': 'Cosine Similarity', 'source': 1},
    {'disp': 'Sens',    'csv_name': 'Sensitivity',       'source': 1},
    {'disp': 'TCAV',    'csv_name': 'TCAV Score',        'source': 1}
]

def format_p_value(val):
    """
    Formats p-value for LaTeX.
    - If p < 0.001: Uses scientific notation (e.g., 1.23e-05)
    - Otherwise: Uses 3 decimal places (e.g., 0.045)
    """
    try:
        p = float(val)
        if p < 0.001:
            # Use scientific notation with 1 decimal digit (e.g. 5.4e-92) 
            # or 2 decimal digits (e.g. 5.42e-92).
            # Using .2e gives 1.23e-05
            return f"{p:.2e}"
        return f"{p:.3f}"
    except (ValueError, TypeError):
        return "-"

def format_float(val):
    """Formats standard floats to 3 decimal places."""
    try:
        return f"{float(val):.3f}"
    except (ValueError, TypeError):
        return "-"

def get_layer_data(df, metric_name, col_name, fmt_func):
    """
    Extracts data for layers 0-11 for a specific metric.
    Returns a list of 12 formatted strings.
    """
    results = []
    # Filter dataframe for the specific metric
    # We assume the CSV has a 'Metric' column and 'Layer' column (0 to 11)
    subset = df[df['Metric'] == metric_name]
    
    for layer in range(12):
        row = subset[subset['Layer'] == layer]
        if not row.empty:
            val = row.iloc[0][col_name]
            results.append(fmt_func(val))
        else:
            results.append("-")
    return results

def generate_table(experiments):
    """
    experiments: List of tuples ("Concept Name", "path_to_acc_csv", "path_to_tcav_csv")
    """
    latex_lines = []
    
    # --- 1. Header ---
    header = r"""\begin{table}[ht]
\centering
\resizebox{\textwidth}{!}{%
\begin{tabular}{llcccccccccccc}
\toprule
\textbf{Concept} & \textbf{Metric} & \textbf{L0} & \textbf{L1} & \textbf{L2} & \textbf{L3} & \textbf{L4} & \textbf{L5} & \textbf{L6} & \textbf{L7} & \textbf{L8} & \textbf{L9} & \textbf{L10} & \textbf{L11} \\
\midrule"""
    latex_lines.append(header)

    # --- 2. Process Null (Using the first experiment as the source of truth) ---
    if not experiments:
        print("No experiments provided.")
        return

    first_concept = experiments[0]
    # Check if files exist for the first concept to safely get Nulls
    try:
        dfs = [pd.read_csv(first_concept[1]), pd.read_csv(first_concept[2])]
        
        latex_lines.append(r"\multirow{4}{*}{Null}")
        
        for m in METRIC_CONFIG:
            # Load the correct DF (Performance or Interpretability)
            current_df = dfs[m['source']]
            
            # Get Null Means
            null_vals = get_layer_data(current_df, m['csv_name'], 'Null_Mean', format_float)
            latex_lines.append(f" & {m['disp']} & " + " & ".join(null_vals) + r" \\")
            
            # Optional: Add divider between Null rows if desired (from your snippet)
            if m != METRIC_CONFIG[-1]:
                 latex_lines.append(r" \cmidrule(l){2-14}")
        
        latex_lines.append(r"\midrule")
        
    except FileNotFoundError as e:
        print(f"Error loading initial files for Null distribution: {e}")
        return

    # --- 3. Process Each Concept ---
    for concept_name, acc_path, tcav_path in experiments:
        # Load Dataframes
        try:
            df_perf = pd.read_csv(acc_path)
            df_interp = pd.read_csv(tcav_path)
            dfs = [df_perf, df_interp]
        except FileNotFoundError as e:
            print(f"Error loading files for {concept_name}: {e}")
            continue

        # Clean concept name for LaTeX (escape underscores)
        clean_name = concept_name.replace("_", r"\_")
        
        # Concept Multirow Header
        latex_lines.append(r"\multirow{12}{*}{\shortstack[l]{" + clean_name + r"}}")
        
        for i, m in enumerate(METRIC_CONFIG):
            current_df = dfs[m['source']]
            
            # 1. Real Mean
            real_vals = get_layer_data(current_df, m['csv_name'], 'Real_Mean', format_float)
            latex_lines.append(f" & {m['disp']} & " + " & ".join(real_vals) + r" \\")
            
            # 2. P-Value (Grey, Footnotesize, Scientific Notation if small)
            p_vals = get_layer_data(current_df, m['csv_name'], 'P_Adjusted', format_p_value)
            p_str = " & ".join([fr"\footnotesize\color{{gray}}{v}" for v in p_vals])
            latex_lines.append(r" & \footnotesize\color{gray}\textit{p-val} & " + p_str + r" \\")
            
            # 3. Cohen's d (Grey, Footnotesize)
            d_vals = get_layer_data(current_df, m['csv_name'], 'Effect_Size_Cohen_d', format_float)
            d_str = " & ".join([fr"\footnotesize\color{{gray}}{v}" for v in d_vals])
            latex_lines.append(r" & \footnotesize\color{gray}\textit{Cohen's d} & " + d_str + r" \\")
            
            # Divider (unless it's the last metric)
            if i < len(METRIC_CONFIG) - 1:
                latex_lines.append(r" \cmidrule(l){2-14}")
        
        # Add a separator between concepts
        if experiments.index((concept_name, acc_path, tcav_path)) < len(experiments) - 1:
            latex_lines.append(r"\midrule")

    # --- 4. Footer ---
    footer = r"""\bottomrule
\end{tabular}%
}
\caption{Summary of metrics. Statistics (p-val, Cohen's d) are shown in grey.}
\label{tab:metrics_summary}
\end{table}"""
    latex_lines.append(footer)
    
    # --- 5. Output ---
    full_latex = "\n".join(latex_lines)
    
    # Save to file
    with open("final_metrics_table.tex", "w") as f:
        f.write(full_latex)
    

    print(full_latex) # Uncomment to print to console

# ==========================================
#              USER INPUTS
# ==========================================

# List of tuples: (Concept Name, Path to Acc/AUC CSV, Path to TCAV/Sens CSV)
EXPERIMENTS = [
    (
        "Circling Open", 
        "/Volumes/T7/Concepts/TCAV/circling/concepts/circling_open/3_real_vs_null_cavs/stats_summary_concept_set_filter.csv",
        "/Volumes/T7/Concepts/TCAV/circling/concepts/circling_open/5_real_vs_null_tcav_scores/stats_summary_concept_set_filter.csv",
    ),
    (
        "Circling Closed", 
        "/Volumes/T7/Concepts/TCAV/circling/concepts/circling_closed/3_real_vs_null_cavs/stats_summary_concept_set_filter.csv",
        "/Volumes/T7/Concepts/TCAV/circling/concepts/circling_closed/5_real_vs_null_tcav_scores/stats_summary_concept_set_filter.csv",
    ),
    (
        "FG Visual Feedback",
        "/Volumes/T7/Concepts/TCAV/circling/concepts/fg_visual_feedback/3_real_vs_null_cavs/stats_summary_concept_set_filter.csv",
        "/Volumes/T7/Concepts/TCAV/circling/concepts/fg_visual_feedback/5_real_vs_null_tcav_scores/stats_summary_concept_set_filter.csv",
    ),
    (
        "FG No Visual Feedback",
        "/Volumes/T7/Concepts/TCAV/circling/concepts/fg_no_visual_feedback/3_real_vs_null_cavs/stats_summary_concept_set_filter.csv",
        "/Volumes/T7/Concepts/TCAV/circling/concepts/fg_no_visual_feedback/5_real_vs_null_tcav_scores/stats_summary_concept_set_filter.csv",
    ),
]

if __name__ == "__main__":
    generate_table(EXPERIMENTS)

In [ ]:
import pandas as pd
import numpy as np
import os

# --- CONFIGURATION ---
# Define the order of metrics and which CSV they come from
# keys: 'Display Name', 'CSV Metric Name', 'Source Index' (0=Perf CSV, 1=Interp CSV)
METRIC_CONFIG = [
    {'disp': 'CAV Acc', 'csv_name': 'Accuracy',          'source': 0},
    {'disp': 'Cos Sim', 'csv_name': 'Cosine Similarity', 'source': 1},
    {'disp': 'Sens',    'csv_name': 'Sensitivity',       'source': 1},
    {'disp': 'TCAV',    'csv_name': 'TCAV Score',        'source': 1}
]

def format_p_value(val):
    """
    Formats p-value for LaTeX.
    - If p < 0.001: Uses scientific notation (e.g., 1.23e-05)
    - Otherwise: Uses 3 decimal places (e.g., 0.045)
    """
    try:
        p = float(val)
        if p < 0.001:
            # Use scientific notation with 1 decimal digit (e.g. 5.4e-92) 
            # or 2 decimal digits (e.g. 5.42e-92).
            # Using .2e gives 1.23e-05
            return f"{p:.2e}"
        return f"{p:.3f}"
    except (ValueError, TypeError):
        return "-"

def format_float(val):
    """Formats standard floats to 3 decimal places."""
    try:
        return f"{float(val):.3f}"
    except (ValueError, TypeError):
        return "-"

def get_layer_data(df, metric_name, col_name, fmt_func):
    """
    Extracts data for layers 0-11 for a specific metric.
    Returns a list of 12 formatted strings.
    """
    results = []
    # Filter dataframe for the specific metric
    # We assume the CSV has a 'Metric' column and 'Layer' column (0 to 11)
    subset = df[df['Metric'] == metric_name]
    
    for layer in range(12):
        row = subset[subset['Layer'] == layer]
        if not row.empty:
            val = row.iloc[0][col_name]
            results.append(fmt_func(val))
        else:
            results.append("-")
    return results

def generate_table(experiments):
    """
    experiments: List of tuples ("Concept Name", "path_to_acc_csv", "path_to_tcav_csv")
    """
    latex_lines = []
    
    # --- 1. Header ---
    header = r"""\begin{table}[ht]
\centering
\resizebox{\textwidth}{!}{%
\begin{tabular}{llcccccccccccc}
\toprule
\textbf{Concept} & \textbf{Metric} & \textbf{L0} & \textbf{L1} & \textbf{L2} & \textbf{L3} & \textbf{L4} & \textbf{L5} & \textbf{L6} & \textbf{L7} & \textbf{L8} & \textbf{L9} & \textbf{L10} & \textbf{L11} \\
\midrule"""
    latex_lines.append(header)

    # --- 2. Process Null (Using the first experiment as the source of truth) ---
    if not experiments:
        print("No experiments provided.")
        return

    first_concept = experiments[0]
    # Check if files exist for the first concept to safely get Nulls
    try:
        dfs = [pd.read_csv(first_concept[1]), pd.read_csv(first_concept[2])]
        
        latex_lines.append(r"\multirow{4}{*}{Null}")
        
        for m in METRIC_CONFIG:
            # Load the correct DF (Performance or Interpretability)
            current_df = dfs[m['source']]
            
            # Get Null Means
            null_vals = get_layer_data(current_df, m['csv_name'], 'Null_Mean', format_float)
            latex_lines.append(f" & {m['disp']} & " + " & ".join(null_vals) + r" \\")
            
            # Optional: Add divider between Null rows if desired (from your snippet)
            if m != METRIC_CONFIG[-1]:
                 latex_lines.append(r" \cmidrule(l){2-14}")
        
        latex_lines.append(r"\midrule")
        
    except FileNotFoundError as e:
        print(f"Error loading initial files for Null distribution: {e}")
        return

    # --- 3. Process Each Concept ---
    for concept_name, acc_path, tcav_path in experiments:
        # Load Dataframes
        try:
            df_perf = pd.read_csv(acc_path)
            df_interp = pd.read_csv(tcav_path)
            dfs = [df_perf, df_interp]
        except FileNotFoundError as e:
            print(f"Error loading files for {concept_name}: {e}")
            continue

        # Clean concept name for LaTeX (escape underscores)
        clean_name = concept_name.replace("_", r"\_")
        
        # Concept Multirow Header
        latex_lines.append(r"\multirow{12}{*}{\shortstack[l]{" + clean_name + r"}}")
        
        for i, m in enumerate(METRIC_CONFIG):
            current_df = dfs[m['source']]
            
            # 1. Real Mean
            real_vals = get_layer_data(current_df, m['csv_name'], 'Real_Mean', format_float)
            latex_lines.append(f" & {m['disp']} & " + " & ".join(real_vals) + r" \\")
            
            # 2. P-Value (Grey, Footnotesize, Scientific Notation if small)
            p_vals = get_layer_data(current_df, m['csv_name'], 'P_Adjusted', format_p_value)
            p_str = " & ".join([fr"\footnotesize\color{{gray}}{v}" for v in p_vals])
            latex_lines.append(r" & \footnotesize\color{gray}\textit{p-val} & " + p_str + r" \\")
            
            # 3. Cohen's d (Grey, Footnotesize)
            d_vals = get_layer_data(current_df, m['csv_name'], 'Effect_Size_Cohen_d', format_float)
            d_str = " & ".join([fr"\footnotesize\color{{gray}}{v}" for v in d_vals])
            latex_lines.append(r" & \footnotesize\color{gray}\textit{Cohen's d} & " + d_str + r" \\")
            
            # Divider (unless it's the last metric)
            if i < len(METRIC_CONFIG) - 1:
                latex_lines.append(r" \cmidrule(l){2-14}")
        
        # Add a separator between concepts
        if experiments.index((concept_name, acc_path, tcav_path)) < len(experiments) - 1:
            latex_lines.append(r"\midrule")

    # --- 4. Footer ---
    footer = r"""\bottomrule
\end{tabular}%
}
\caption{Summary of metrics. Statistics (p-val, Cohen's d) are shown in grey.}
\label{tab:metrics_summary}
\end{table}"""
    latex_lines.append(footer)
    
    # --- 5. Output ---
    full_latex = "\n".join(latex_lines)
    
    # Save to file
    with open("final_metrics_table.tex", "w") as f:
        f.write(full_latex)
    

    print(full_latex) # Uncomment to print to console

# ==========================================
#              USER INPUTS
# ==========================================

# List of tuples: (Concept Name, Path to Acc/AUC CSV, Path to TCAV/Sens CSV)
EXPERIMENTS = [
    (
        "MG Instructed Coordination",
        "/Volumes/T7/Concepts/TCAV/circling/concepts/mg_coordination/3_real_vs_null_cavs/stats_summary_concept_set_filter.csv",
        "/Volumes/T7/Concepts/TCAV/circling/concepts/mg_coordination/5_real_vs_null_tcav_scores/stats_summary_concept_set_filter.csv",
    ),
    (
        "MG Spontaneous Coordination",
        "/Volumes/T7/Concepts/TCAV/circling/concepts/mg_spontaneous/3_real_vs_null_cavs/stats_summary_concept_set_filter.csv",
        "/Volumes/T7/Concepts/TCAV/circling/concepts/mg_spontaneous/5_real_vs_null_tcav_scores/stats_summary_concept_set_filter.csv",
    ),

    (
        "MG Solo",
        "/Volumes/T7/Concepts/TCAV/circling/concepts/mg_solo/3_real_vs_null_cavs/stats_summary_concept_set_filter.csv",
        "/Volumes/T7/Concepts/TCAV/circling/concepts/mg_solo/5_real_vs_null_tcav_scores/stats_summary_concept_set_filter.csv",
    ),
]

if __name__ == "__main__":
    generate_table(EXPERIMENTS)

In [ ]:
import os
from pathlib import Path

def rename_folders():
    # Define the root directory
    # This assumes the script is running in the same directory as the "concepts" folder
    root_dir = Path("/Volumes/T9/Concepts/TCAV_neural/mg_spont-instruct/concepts")

    # Check if 'concepts' folder actually exists
    if not root_dir.exists():
        print("❌ Error: Could not find the 'concepts' folder. Please place this script next to it.")
        return

    print(f"📂 Scanning inside '{root_dir}'...\n")

    # Iterate through everything inside 'concepts'
    for subfolder in root_dir.iterdir():
        # Only process if it is a directory (e.g., alpha_AF3_high)
        if subfolder.is_dir():
            
            # Define the old path and the new path
            old_path = subfolder / "concept_activations"
            new_path = subfolder / "1_concept_activations"

            # Check if the 'concept_activations' folder exists
            if old_path.exists():
                try:
                    # Perform the rename
                    old_path.rename(new_path)
                    print(f"✅ Renamed in {subfolder.name}: 'concept_activations' -> '1_concept_activations'")
                except Exception as e:
                    print(f"❌ Error renaming in {subfolder.name}: {e}")
            
            # Check if it looks like it was already renamed
            elif new_path.exists():
                print(f"ℹ️  Skipped {subfolder.name}: Folder is already named '1_concept_activations'")
            
            else:
                print(f"⚠️  Skipped {subfolder.name}: No 'concept_activations' folder found.")

    print("\n✨ Process complete.")

rename_folders()